# 02 - ML-Based Reservoir Quality Ranking for CO2 Storage
### Niger Delta → CaMI FRS Transfer Learning Concept
**Author:** Umaru Ibe
**Goal:** Train Random Forest + XGBoost to rank Excellent/Good/Marginal/Poor CO2 storage candidates using petrophysical attributes.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# Recreate or load from Notebook 01
np.random.seed(1)
N=800
df = pd.DataFrame({
    'VSH': np.random.beta(2,5,N),
    'PHIE': np.random.normal(0.18,0.06,N).clip(0.05,0.32),
    'PERM_mD': np.random.lognormal(3,1.2,N),
    'NTG': np.random.uniform(0.3,0.9,N),
    'RT': np.random.normal(25,10,N),
    'DT': np.random.normal(85,12,N)
})

# Label reservoir quality for CO2 storage
def label_quality(row):
    if row['PHIE']>0.20 and row['PERM_mD']>200 and row['VSH']<0.25: return 'Excellent'
    elif row['PHIE']>0.15 and row['PERM_mD']>50: return 'Good'
    elif row['PHIE']>0.10 and row['PERM_mD']>10: return 'Marginal'
    else: return 'Poor'

df['QUALITY'] = df.apply(label_quality, axis=1)
df.head()


In [ ]:
# Train/test split
X = df[['VSH','PHIE','PERM_mD','NTG','RT','DT']]
y = df['QUALITY']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25, random_state=42, stratify=y)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test)))

# XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
print(classification_report(y_test, xgb_model.predict(X_test)))


In [ ]:
# Feature importance + SHAP (key for Calgary interview)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
importances.plot(kind='barh', title='Feature Importance for CO2 Storage Ranking')
plt.show()

# SHAP summary (optional - comment if no shap installed)
# explainer = shap.TreeExplainer(rf)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)


### Transferability to CaMI FRS, Alberta
- Niger Delta high porosity/permeability sands are analogous to Basal Belly River Fm at CaMI
- Workflow retrainable with WCSB logs: adjust VSH cutoffs (Western Canadian shales lower GR)
- Excellent/Good classes correspond to >100 mD, >15% phi threshold used by CaMI Containment & Monitoring Institute for CO2 plume migration
- Next step: Incorporate time-lapse seismic monitoring thresholds (NRMS < 10%) as additional feature

**For Prof. Lawton email:** Reference this repo as proof of reproducible ML + petrophysics + CCS focus.
